> 📒 이 노트북은 [펭귄 데이터로 배우는 인공지능](https://yanghyeokjin33.github.io/Algorithm/) 수업에서 쓰는 실습 노트북이에요. 원본: [https://github.com/YANGHYEOKJIN33/Algorithm](https://github.com/YANGHYEOKJIN33/Algorithm)

# 🤖 k-평균 (k-Means)

**군집 · 비지도학습** — 정답(종) 없이, 가까운 펭귄끼리 k개 묶음으로 나눈다.

    중심 k개 ← 아무 점 k개
    반복: ① 각 점을 가장 가까운 중심의 묶음에  ② 중심을 묶음의 평균으로  ③ 안 움직이면 멈춤

> **Colab 사용법** — 셀 왼쪽의 ▶ 단추를 누르거나 **Shift + Enter**를 누르면 그 셀이 실행돼요.
> 맨 위 셀부터 **차례로** 실행하세요. 코드를 고쳐 보고 싶다면 먼저 **파일 → 드라이브에 사본 저장**을 눌러 내 사본을 만들어 두세요.
> 파이썬을 몰라도 괜찮아요. 코드 위의 설명을 사이트의 의사코드와 한 줄씩 맞춰 읽으면 돼요.

- 비지도학습이라 y(정답)가 없어요.

In [ ]:
import pandas as pd
df = pd.read_csv('https://raw.githubusercontent.com/YANGHYEOKJIN33/Algorithm/main/data/penguins_clean.csv')
X = df[['부리길이', '부리깊이']]          # 정답(종)은 쓰지 않아요!

## 1. 의사코드 그대로 — 사이트와 같은 18마리, 처음 중심 21·160·282번

- d.index(min(d))는 "거리가 가장 짧은 중심의 번호"예요.
- 사이트에서처럼 3번째 되풀이에서 멈춰요.

In [ ]:
ids = [1, 6, 14, 21, 31, 38, 277, 278, 280, 282, 285, 288, 153, 154, 158, 160, 161, 164]
pts = df[df['번호'].isin(ids)][['번호', '부리길이', '부리깊이']].values.tolist()
centers = [[p[1], p[2]] for p in pts if p[0] in (21, 160, 282)]

for it in range(1, 20):
    # ① 배정: 가장 가까운 중심 찾기
    groups = [[] for _ in centers]
    for p in pts:
        d = [((p[1] - c[0])**2 + (p[2] - c[1])**2) ** 0.5 for c in centers]
        groups[d.index(min(d))].append(p)
    # ② 이동: 묶음의 평균으로
    new = [[sum(p[1] for p in g) / len(g), sum(p[2] for p in g) / len(g)] for g in groups]
    print(f'{it}번째:', [[round(v, 2) for v in c] for c in new])
    if new == centers:            # ③ 움직이지 않으면 멈춤
        break
    centers = new

## 2. scikit-learn KMeans로 전체 데이터 묶기

부리길이(32~60mm)는 부리깊이(13~22mm)보다 값의 폭이 훨씬 넓어요. 그대로 두면 부리길이가 거리를 거의 혼자 정해 버려요.
그래서 `StandardScaler`로 두 속성을 평균 0, 퍼짐 1이 되도록 크기를 맞춘 뒤에 묶어요.

- n_init=10은 처음 중심을 10번 다르게 골라 보고, 그중 가장 좋은 결과를 쓰라는 뜻이에요. 처음 중심에 따라 결과가 달라지거든요.
- crosstab으로 묶음과 실제 종을 견줘요. 정답을 보지 않았는데도 대부분 종끼리 모였어요.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
Xs = scaler.fit_transform(X)                 # 크기 맞추기
model = KMeans(n_clusters=3, n_init=10, random_state=0)
model.fit(Xs)                                # 정답 없이 X만!
df['묶음'] = model.labels_                   # 점마다 묶음 번호(0, 1, 2)
pd.crosstab(df['종'], df['묶음'])

- 크기를 맞춘 쪽이 실제 종과 훨씬 잘 맞아요. k-최근접 이웃이나 k-평균처럼 거리로 판단하는 알고리즘에서는 속성의 크기를 꼭 맞춰야 해요.

In [ ]:
# 크기를 맞추지 않으면 어떨까? X 그대로 같은 일을 해 보기
raw = KMeans(n_clusters=3, n_init=10, random_state=0).fit(X)
def agree(labels):   # 묶음마다 가장 많은 종을 고른다면 몇 %가 맞을까
    return pd.crosstab(df['종'], labels).max(axis=1).sum() / len(df)
print(f'크기 맞춤: {agree(model.labels_):.0%}   그대로: {agree(raw.labels_):.0%}')

- 그래프의 제목이나 축 이름에 쓴 한글이 깨지지 않도록 글꼴 도구를 설치해요. 처음 한 번은 1~2분쯤 걸릴 수 있어요.

In [ ]:
!pip install -q koreanize-matplotlib
try:
    import koreanize_matplotlib  # 그래프에 한글 글꼴을 입혀 줘요
except ImportError:
    print('한글 글꼴을 불러오지 못했어요. 그래프 글자가 네모로 보일 수 있어요.')
import matplotlib.pyplot as plt

- 점의 색은 묶음, 빨간 X는 중심을 나타내요. 중심은 크기를 맞춘 단위로 계산했기 때문에 원래 mm 단위로 되돌려서 그려요.

In [ ]:
centers = scaler.inverse_transform(model.cluster_centers_)   # 중심을 원래 단위(mm)로 되돌리기
plt.scatter(df['부리길이'], df['부리깊이'], c=df['묶음'], cmap='viridis', alpha=0.7)
plt.scatter(centers[:, 0], centers[:, 1], marker='X', s=300, c='red', label='중심')
plt.xlabel('부리길이 (mm)'); plt.ylabel('부리깊이 (mm)'); plt.legend()
plt.show()

## 🚀 도전 — k를 몇으로 할까?

`inertia_`는 점마다 자기 중심까지의 거리를 제곱해 모두 더한 값이에요. k를 늘려 가며 그래프를 그리고, 줄어드는 폭이 갑자기 작아지는 "팔꿈치" 지점을 찾아보세요.

- k가 커지면 inertia는 언제나 줄어들어요. 그래서 가장 작은 곳이 아니라 "줄어드는 폭이 확 작아지는 곳"을 골라요.

In [ ]:
ks = range(1, 8)
inertias = [KMeans(n_clusters=k, n_init=10, random_state=0).fit(Xs).inertia_ for k in ks]
plt.plot(list(ks), inertias, marker='o')
plt.xlabel('k (묶음 수)'); plt.ylabel('inertia')
plt.title('팔꿈치 방법')
plt.show()